In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import argparse
import yaml

from environment import SailingEnv, Config
from vector_field import VecField
from map_elements import Checkpoint

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument('--config', type=str, default='../config.yaml', help="Path to config file")
args = parser.parse_args()

try:
    with open(args.config, 'r') as f:
        cfg = yaml.safe_load(f)
except FileNotFoundError:
    print(f"Config file {args.config} not found. Exiting.")
    raise SystemExit(1)


configs = Config(

    max_speed = 10.0,
    sail_rotation_speed = 0.1,
    boat_rotation_speed = 0.05,
    initial_sail_angle = 0.0,
    initial_boat_angle = 0.0, 
    initial_position = np.array([0.0, 0.0]),

    map_width = 100,
    map_height = 100,
    water_friction = 3.0,

    dt = 0.1,
    max_steps = 2000000,

    # Rendering
    window_width = 600,
    window_height = 600,
    render_fps = 30)

In [4]:
vector_field = VecField(configs.map_width, configs.map_height, None)
goal = Checkpoint(np.array([80.0, 80.0]), radius=5.0, number=3)
cp1 = Checkpoint(np.array([20.0, 20.0]), radius=5.0, number=1)
cp2 = Checkpoint(np.array([50.0, 50.0]), radius=5.0, number=2)
checkpoints = [cp1, cp2]

env = SailingEnv(configs, vector_field, goal=goal, checkpoints=checkpoints, render_mode="human")
observation, info = env.reset()

observation

/home/lorenzo/STORAGE/UNI/MAGISTRALE/REINFORCEMENT LEARNING/Exam project/Barche-Barche-Barche/.rl_venv/lib64/python3.11/site-packages/gymnasium/spaces/box.py:231: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/home/lorenzo/STORAGE/UNI/MAGISTRALE/REINFORCEMENT LEARNING/Exam project/Barche-Barche-Barche/.rl_venv/lib64/python3.11/site-packages/gymnasium/spaces/box.py:297: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


{'boat_position': array([0., 0.]),
 'boat_velocity': array([0., 0.]),
 'sail_angle': 0.0,
 'boat_angle': 0.0,
 'wind_vector': array([0., 0.]),
 'next_checkpoint_relative': array([20.        , 20.        , 28.28427125]),
 'next_next_checkpoint_relative': array([50.        , 50.        , 70.71067812]),
 'goal_relative': array([ 80.        ,  80.        , 113.13708499])}

In [ ]:
episode_over = False
total_reward = 0

while not episode_over:
    action = env.action_space.sample() 

    observation, reward, terminated, truncated, info = env.step(action)

    total_reward += reward
    episode_over = terminated or truncated
    print(f"Step reward: {reward}, Total reward: {total_reward}, Episode over: {episode_over}")
    print(f"Boat position: {observation['boat_position']}, Boat velocity: {observation['boat_velocity']}, Boat angle: {observation['boat_angle']}")


Step reward: 0.00883894361291747, Total reward: 0.00883894361291747, Episode over: False
Boat position: [ 1.99994033e-03 -1.54489486e-05], Boat velocity: [ 0.0199994  -0.00015449], Boat angle: -0.007724551111459733, Sail angle: 0.047833997011184695
Step reward: 0.008839152297868618, Total reward: 0.01767809591078609, Episode over: False
Boat position: [ 0.00599013 -0.00022811], Boat velocity: [ 0.03990193 -0.00212666], Boat angle: -0.09876898303627968, Sail angle: 0.049379386194050315
Step reward: 0.008839455092849542, Total reward: 0.026517551003635634, Episode over: False
Boat position: [ 0.01196102 -0.00071802], Boat velocity: [ 0.05970884 -0.00489909], Boat angle: -0.13906945362687112, Sail angle: 0.06663162279874087
Step reward: 0.00883984542235838, Total reward: 0.03535739642599402, Episode over: False
Boat position: [ 0.0198956  -0.00158725], Boat velocity: [ 0.07934584 -0.00869228], Boat angle: -0.1908154644072056, Sail angle: 0.01777119506150484
Step reward: 0.0088403207361787

KeyboardInterrupt: 

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import gymnasium as gym
import matplotlib.pyplot as plt

# CRITICAL SPEED FIX: 
# Disable CPU multithreading to eliminate overhead on tiny networks.
torch.set_num_threads(1)

class PPOActorCriticNetwork(nn.Module):
    r"""
    A shared-body neural network for the PPO algorithm.
    Maps the 4D state vector to a policy distribution and a state-value estimate.
    """
    def __init__(self, state_dim: int = 4, action_dim: int = 2, hidden_dim: int = 64):
        super(PPOActorCriticNetwork, self).__init__()
        
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.Tanh(), # Tanh is generally preferred over ReLU for PPO feature extractors
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh()
        )
        
        self.actor_head = nn.Sequential(
            nn.Linear(hidden_dim, action_dim),
            nn.Softmax(dim=-1)
        )
        
        self.critic_head = nn.Linear(hidden_dim, 1)

    def forward(self, state: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        r"""
        Forward pass computing action probabilities and V(s) simultaneously.
        """
        features = self.shared(state)
        action_probs = self.actor_head(features)
        state_value = self.critic_head(features)
        return action_probs, state_value


class FastPPOAgent:
    r"""
    Implements Proximal Policy Optimization (PPO-Clip).
    Collects rollouts and performs multiple epochs of optimization using the clipped surrogate objective.
    """
    def __init__(self, state_dim: int = 4, action_dim: int = 2, gamma: float = 0.99, lr: float = 2e-3, 
                 clip_ratio: float = 0.2, ppo_epochs: int = 4):
        self.gamma = gamma
        self.clip_ratio = clip_ratio
        self.ppo_epochs = ppo_epochs
        
        self.net = PPOActorCriticNetwork(state_dim, action_dim)
        self.optimizer = optim.Adam(self.net.parameters(), lr=lr)
        
        # Buffer to store trajectories for the batch update
        self.buffer = []

    def get_action(self, state: np.ndarray) -> tuple[int, float]:
        r"""
        Samples an action and returns both the action index and its log probability.
        The log probability is required later to calculate the PPO ratio.
        """
        s_tensor = torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            probs, _ = self.net(s_tensor)
            
        dist = Categorical(probs)
        action = dist.sample()
        
        return action.item(), dist.log_prob(action).item()

    def store_transition(self, transition: tuple) -> None:
        r"""
        Stores a (state, action, reward, next_state, done, log_prob) tuple in the buffer.
        """
        self.buffer.append(transition)

    def update(self) -> None:
        r"""
        Executes the PPO update algorithm on the collected batch of trajectories.
        Computes Advantages, and performs multiple epochs of clipped gradient descent.
        """
        if len(self.buffer) == 0:
            return
            
        # 1. Unpack the buffer
        states = torch.tensor(np.array([t[0] for t in self.buffer]), dtype=torch.float32)
        actions = torch.tensor(np.array([t[1] for t in self.buffer]), dtype=torch.float32)
        rewards = [t[2] for t in self.buffer]
        dones = [t[4] for t in self.buffer]
        old_log_probs = torch.tensor(np.array([t[5] for t in self.buffer]), dtype=torch.float32)
        
        # 2. Calculate Discounted Returns (Monte Carlo style for simplicity)
        returns = []
        discounted_sum = 0
        for reward, done in zip(reversed(rewards), reversed(dones)):
            if done:
                discounted_sum = 0
            discounted_sum = reward + (self.gamma * discounted_sum)
            returns.insert(0, discounted_sum)
            
        returns = torch.tensor(returns, dtype=torch.float32)
        
        # Normalize returns for stability
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        # 3. Perform PPO Epochs
        for _ in range(self.ppo_epochs):
            # Recalculate probabilities and values under the CURRENT, continually updating network
            probs, state_values = self.net(states)
            state_values = state_values.squeeze()
            
            dist = Categorical(probs)
            curr_log_probs = dist.log_prob(actions)
            
            # Calculate Advantage
            # Advantage must be detached so gradients don't flow backward through the target calculation
            advantages = returns - state_values.detach()
            
            # 4. Calculate PPO Ratio: r(theta) = pi_new / pi_old = exp(log_new - log_old)
            ratios = torch.exp(curr_log_probs - old_log_probs)
            
            # 5. Calculate Clipped Surrogate Objective
            surr1 = ratios * advantages
            surr2 = torch.clamp(ratios, 1.0 - self.clip_ratio, 1.0 + self.clip_ratio) * advantages
            
            # Actor Loss: maximize surrogate (minimize negative surrogate)
            actor_loss = -torch.min(surr1, surr2).mean()
            
            # Critic Loss: MSE between V(s) and returns
            critic_loss = nn.MSELoss()(state_values, returns)
            
            # Entropy Bonus (optional, encourages exploration)
            entropy = dist.entropy().mean()
            
            # Total Loss formulation
            loss = actor_loss + 0.5 * critic_loss - 0.01 * entropy
            
            # Backpropagation
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            
        # Clear the buffer after the batch update is complete
        self.buffer.clear()

def train_ppo_agent(agent, env: gym.Env, n_episodes: int = 1500, update_timestep: int = 2000) -> list:
    r"""
    Executes the PPO training loop.
    Collects a fixed number of timesteps across episodes before triggering the epoch update.
    """
    returns = np.zeros(n_episodes)
    timestep_counter = 0
    
    for i in range(n_episodes):
        state, _ = env.reset()
        done = False
        truncated = False
        
        while not (done or truncated):
            action, log_prob = agent.get_action(state)
            next_state, reward, done, truncated, _ = env.step(action)
            
            is_terminal = done and not truncated
            
            # Store data in agent's rollout buffer
            agent.store_transition((state, action, reward, next_state, is_terminal, log_prob))
            
            state = next_state
            returns[i] += reward
            timestep_counter += 1
            
            # Trigger PPO update if we have collected enough timesteps
            if timestep_counter % update_timestep == 0:
                agent.update()
                
        if (i + 1) % 500 == 0:
            print(f"[PPO-Clip] Episode {i+1:4d} | Last Return: {returns[i]:.0f}")
            
    return returns